In [4]:
import numpy as np
import json
import sys
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sem_proj.data.datasets import BoasSequenceDataset
from sem_proj.data.preprocessing import PreprocessingConfig
from sem_proj.models.model_factory import SSLEpochTransformerConv1D_v2, SequenceGRUClassifier
from sem_proj.data.boa_loader import build_pid_mappings

CHECKPOINT_LEOMED_DIR = PROJECT_ROOT / "checkpoints_leomed"
JSON_DIR = PROJECT_ROOT / "reports" / "metrics"
TARGET_DIR = PROJECT_ROOT / "plots"

In [ ]:
### check if stronger MLP v1 results are better than with the previous MLP
ctxfree_finetuned_stronger_MLP_v1_json = JSON_DIR / "ctxfree_finetuning_results_val_step_stronger_MLP_v1.json"
ctxfree_fullysupervised_stronger_MLP_v1_json = JSON_DIR / "ctxfree_fullysupervised_results_val_step_stronger_MLP_v1.json"
with open(ctxfree_finetuned_stronger_MLP_v1_json, 'r') as f:
    ctxfree_finetuned_stronger_MLP_v1 = json.load(f)
with open(ctxfree_fullysupervised_stronger_MLP_v1_json, 'r') as f:
    ctxfree_fullysupervised_stronger_MLP_v1 = json.load(f)
mf1_finetuned = []
mf1_fullysuperv = []

for key in ctxfree_finetuned_stronger_MLP_v1:
    nested_dict = ctxfree_finetuned_stronger_MLP_v1[key]
    mf1_score = nested_dict['stage1_mf1']
    mf1_finetuned.append(mf1_score)
for key in ctxfree_fullysupervised_stronger_MLP_v1:
    nested_dict = ctxfree_fullysupervised_stronger_MLP_v1[key]
    mf1_score = nested_dict['stage1_mf1']
    mf1_fullysuperv.append(mf1_score)

p = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
p = [x * 100 for x in p]  # convert to percentage

plt.figure(figsize=(10, 6))
plt.plot(p, mf1_finetuned, marker='x', linewidth=2, markersize=8, label='SSL Pretrain + FT', color='green')
plt.plot(p, mf1_fullysuperv, marker='o', linewidth=2, markersize=8, label='Supervised from Scratch', color='red')
plt.xlabel('\nPercentage of labeled training data (%)', fontsize=18)
plt.ylabel('Macro F1 Score\n', fontsize=18)
plt.legend(fontsize=16)
plt.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.grid(True)
# plt.show()
plt.savefig(TARGET_DIR / 'ctxfree_finetuning_vs_fullysupervised_varying_p_stronger_MLP_v1.pdf', dpi=300, bbox_inches='tight')
